[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yotamc19/Image-processing-project/blob/main/notebooks/milestone3_final.ipynb)

# Milestone 3: Fine-Tuning, Test-Set Evaluation & Inference

Final milestone. Builds on the M2 STN-CRNN:

1. **Fine-tune** the STN-CRNN from the M2 weights at a lower learning rate.
2. **Evaluate on the held-out test split** (final, unbiased numbers — M1/M2
   only reported validation).
3. **Inference pipeline** demo (`src/inference.py`): image → text.

The M2 base checkpoint is regenerated here if absent, then fine-tuned; on a
T4 the full run is ~1.5h (base ~1h + fine-tune ~20m + eval).

## 1. Setup

In [ ]:
import os, sys
if not os.path.exists('Image-processing-project'):
    !git clone https://github.com/yotamc19/Image-processing-project.git
%cd Image-processing-project
!pip install -q -r requirements.txt
sys.path.insert(0, '.')

In [ ]:
# QUICK_TEST=True runs the full M3 pipeline on a small subset (~2 min) to verify wiring.
# Set to False for the real fine-tuning + test evaluation.
QUICK_TEST = True
BASE_CONFIG     = 'configs/smoke_test_m2.yaml' if QUICK_TEST else 'configs/milestone2_stn.yaml'
FINETUNE_CONFIG = 'configs/smoke_test_m3.yaml' if QUICK_TEST else 'configs/milestone3_finetune.yaml'
print(f'Base config:     {BASE_CONFIG}')
print(f'Finetune config: {FINETUNE_CONFIG}')

## 2. STN-CRNN Base (regenerate M2 weights if needed)

Fine-tuning needs the M2 weights as a starting point. If `checkpoints_m2/best_model.pt` already exists (e.g. from a live session), this is skipped;
otherwise the STN-CRNN base is trained first.

In [ ]:
from src.train import train as train_model

if not os.path.exists('checkpoints_m2/best_model.pt'):
    print('No M2 base checkpoint found — training STN-CRNN base first...')
    base_results = train_model(BASE_CONFIG, checkpoint_dir='checkpoints_m2')
else:
    print('Found existing checkpoints_m2/best_model.pt — skipping base training.')
    base_results = None

## 3. Fine-Tune

Continue from the M2 weights at lr 1e-4 (10x lower) for a short schedule.
Early-stops on val CER. Best fine-tuned model saved to `checkpoints_m3/`.

In [ ]:
m3_results = train_model(FINETUNE_CONFIG, checkpoint_dir='checkpoints_m3')

## 4. Final Test-Set Evaluation

Score both the pre-fine-tune (M2 base) and fine-tuned (M3) models on the
**test** split — the final, held-out numbers.

In [ ]:
import torch, yaml
from src.dataset import LabelEncoder
from src.model import CRNN
from src.evaluate import evaluate_test_set, print_report
from src.utils import load_checkpoint

with open(FINETUNE_CONFIG) as f:
    config = yaml.safe_load(f)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
encoder = LabelEncoder()

# Fine-tuned M3 model
m3 = CRNN(num_classes=len(encoder), use_stn=True).to(device)
m3_ckpt = load_checkpoint('checkpoints_m3/best_model.pt', m3)
m3_val_cer, m3_val_wer = m3_ckpt['metrics']['cer'], m3_ckpt['metrics']['wer']
print(f"M3 fine-tuned: best val CER {m3_val_cer:.4f} | WER {m3_val_wer:.4f} (epoch {m3_ckpt['epoch']+1})")

m3_test = evaluate_test_set(m3, config, encoder, device, split='test')
print_report(m3_test)

# M2 base (pre-fine-tune) on test, for comparison
m2 = CRNN(num_classes=len(encoder), use_stn=True).to(device)
load_checkpoint('checkpoints_m2/best_model.pt', m2)
m2_test = evaluate_test_set(m2, config, encoder, device, split='test')
print(f"\nTest mean CER  — M2 base: {m2_test['mean_cer']:.4f} | M3 tuned: {m3_test['mean_cer']:.4f}")
print(f"Test mean WER  — M2 base: {m2_test['mean_wer']:.4f} | M3 tuned: {m3_test['mean_wer']:.4f}")

## 5. Inference Pipeline Demo

`src/inference.py` loads a checkpoint and transcribes an image. Here it runs
on random test images (CLI equivalent: `python -m src.inference --checkpoint
checkpoints_m3/best_model.pt --image word.png`).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from src.inference import predict
from src.dataset import load_iam_splits

_, _, test_hf = load_iam_splits()
idxs = np.random.choice(len(test_hf), 8, replace=False)
fig, axes = plt.subplots(8, 1, figsize=(12, 16))
for ax, i in zip(axes, idxs):
    s = test_hf[int(i)]
    pred = predict(m3, s['image'], encoder, device)
    ax.imshow(s['image'], cmap='gray')
    ax.set_title(f"GT: '{s['text']}'   |   Pred: '{pred}'")
    ax.axis('off')
plt.tight_layout(); plt.show()

## 6. Final Results Across Milestones

In [ ]:
print('=== Final Results Across All Milestones ===')
print(f'{"Model":<26}{"Val CER":>10}{"Val WER":>10}{"Test CER":>10}{"Test WER":>10}')
print('-' * 66)
print(f'{"M1 baseline CRNN":<26}{0.1065:>10.4f}{0.2460:>10.4f}{"—":>10}{"—":>10}')
print(f'{"M2 STN-CRNN + aug":<26}{0.0867:>10.4f}{0.2012:>10.4f}{m2_test["mean_cer"]:>10.4f}{m2_test["mean_wer"]:>10.4f}')
print(f'{"M3 fine-tuned":<26}{m3_val_cer:>10.4f}{m3_val_wer:>10.4f}{m3_test["mean_cer"]:>10.4f}{m3_test["mean_wer"]:>10.4f}')